# Monte Carlo Simulation - Companion Notebook (parts 1 and 2)

**This notebook is one half of a pair.** It carries every piece of code from parts 1 and 2 of the
[Monte Carlo guide](https://statexampro.com/lessons/monte-carlo/) on Stat Exam Pro. The lessons do the explaining; this does the running.

Read a section there, run the matching one here, then poke at it - change an `N`, change a seed, break something on
purpose. Probability is a subject where everything makes sense until you sit down to write your own simulation, which
is exactly why neither half works as well alone.

| The lesson | The code |
|---|---|
| [Part 1 - Monte Carlo from first principles](https://statexampro.com/lessons/monte-carlo/first-principles/) | sections 2-5 below: the dartboard, the law of large numbers, standard errors, seeds |
| [Part 2 - The recipe, and two real projects](https://statexampro.com/lessons/monte-carlo/recipe-and-projects/) | sections 6-9 below: building distributions by hand, the DCF, power by simulation |

Parts 3 and 4 (paths, option pricing, variance reduction, MCMC) have their own
[pro notebook](https://colab.research.google.com/github/StatExamPro/statexampro.com/blob/main/notebooks/monte-carlo-pro.ipynb).

Requirements: `numpy`, `scipy`, `matplotlib` - all preinstalled in Colab, nothing to set up. Runtime: ~1-2 minutes.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams["figure.figsize"] = (9, 4.5)
plt.rcParams["axes.grid"] = True

rng = np.random.default_rng(42)   # our master generator (Chapter 5 explains why)
print(np.__version__)

## Section 2 — Your First Simulation: Estimating π

The move: express π as a probability. A dart uniform on the unit square lands inside the quarter circle with probability π/4. So π ≈ 4 × (fraction of darts inside).
> Explained in the guide: [Part 1 - estimating pi by throwing darts](https://statexampro.com/lessons/monte-carlo/first-principles/#darts)


In [ ]:
N = 1_000_000
x = rng.uniform(0, 1, size=N)
y = rng.uniform(0, 1, size=N)
inside = (x**2 + y**2) <= 1.0
pi_hat = 4 * inside.mean()
print(f"pi_hat = {pi_hat:.6f}   true pi = {np.pi:.6f}   error = {pi_hat - np.pi:+.6f}")

**Experiment A — one run, growing N.** Watch the estimate wander toward π. Convergence is a drunkard weaving home, not a train on rails — local worsening is normal.

In [ ]:
r = np.random.default_rng(0)
X = r.uniform(size=10**7); Y = r.uniform(size=10**7)
hits = ((X**2 + Y**2) <= 1.0)
running_pi = 4 * np.cumsum(hits) / np.arange(1, len(hits) + 1)

n_axis = np.arange(1, len(hits) + 1)
plt.semilogx(n_axis[99:], running_pi[99:], lw=1)
plt.axhline(np.pi, color="k", ls="--", label="true π")
plt.xlabel("N (darts)"); plt.ylabel("estimate"); plt.legend()
plt.title("Experiment A: one estimate as N grows"); plt.show()

for n in [100, 10_000, 1_000_000, 10_000_000]:
    print(f"N = {n:>10,d}   pi_hat = {running_pi[n-1]:.5f}")

**Experiment B — many runs, fixed N.** 2,000 independent repetitions at N = 1,000. The histogram of estimates is the answer to *"if I reran this, how different would my number be?"* Your estimate is a random variable; this is its distribution.

In [ ]:
reps, N = 2000, 1000
r = np.random.default_rng(1)
u = r.uniform(size=(reps, N, 2))
estimates = 4 * ((u**2).sum(axis=2) <= 1.0).mean(axis=1)

plt.hist(estimates, bins=40, density=True, alpha=0.7)
plt.axvline(np.pi, color="k", ls="--", label="true π")
plt.xlabel("π estimate (N=1000 each)"); plt.ylabel("density"); plt.legend()
plt.title("Experiment B: distribution of the estimator")
plt.show()
print(f"mean of estimates = {estimates.mean():.4f}  (unbiased!)")
print(f"sd   of estimates = {estimates.std(ddof=1):.4f}")

**Experiment C — the square-root law.** Typical error vs N on log-log axes: a straight line of slope −1/2. Ten times the accuracy costs a hundred times the darts.

In [ ]:
Ns = [10**k for k in range(2, 8)]
r = np.random.default_rng(2)
typical_err = []
for n in Ns:
    reps = int(np.clip(2_000_000 // n, 20, 500))   # fewer reps for huge n, enough for a trend
    est = np.empty(reps)
    for i in range(reps):                           # one replication at a time (memory-friendly)
        u = r.uniform(size=(n, 2))
        est[i] = 4 * ((u**2).sum(axis=1) <= 1.0).mean()
    typical_err.append(np.sqrt(np.mean((est - np.pi) ** 2)))   # RMS error

plt.loglog(Ns, typical_err, "o-", label="measured RMS error")
c = typical_err[0] * np.sqrt(Ns[0])
plt.loglog(Ns, c / np.sqrt(Ns), "k--", label=r"$\propto 1/\sqrt{N}$")
plt.xlabel("N"); plt.ylabel("typical error"); plt.legend()
plt.title("Experiment C: error ∝ N^(−1/2)"); plt.show()

## Section 3 — The Law of Large Numbers, Watched Live

Running means for four distributions. Well-behaved ones funnel neatly onto the truth. The high-volatility lognormal converges in ugly stair-steps (rare huge draws yank the average). The Cauchy — no finite mean — never converges at all.
> Explained in the guide: [Part 1 - why it works: the law of large numbers](https://statexampro.com/lessons/monte-carlo/first-principles/#lln)


In [ ]:
r = np.random.default_rng(3)
N = 200_000
n_axis = np.arange(1, N + 1)

fig, axes = plt.subplots(2, 2, figsize=(11, 7))
cases = [
    ("Uniform(0,1), mean 0.5", lambda: r.uniform(size=N), 0.5),
    ("Normal(0,1), mean 0",    lambda: r.normal(size=N), 0.0),
    ("Lognormal(0, 1.5) — heavy right skew", lambda: r.lognormal(0, 1.5, size=N), np.exp(1.5**2 / 2)),
    ("Cauchy — NO finite mean", lambda: r.standard_cauchy(size=N), None),
]
for ax, (title, gen, mu) in zip(axes.ravel(), cases):
    for _ in range(3):                       # three independent runs each
        xs = gen()
        ax.plot(n_axis, np.cumsum(xs) / n_axis, lw=0.8)
    if mu is not None:
        ax.axhline(mu, color="k", ls="--")
    ax.set_xscale("log"); ax.set_title(title, fontsize=10)
fig.suptitle("Running means: LLN at work (and one crime scene)")
fig.tight_layout(); plt.show()

## Section 4 — CLT, Standard Errors, and Confidence Intervals

**The bell curve assembles itself.** Take a comically skewed distribution (exponential, even a mixture). The distribution of the *sample mean* becomes normal as N grows — regardless of the parent's shape.
> Explained in the guide: [Part 1 - how wrong are you? the square-root law](https://statexampro.com/lessons/monte-carlo/first-principles/#sqrt-law)


In [ ]:
r = np.random.default_rng(4)
parent = lambda size: np.where(r.uniform(size=size) < 0.9,
                               r.exponential(1.0, size=size),
                               r.exponential(15.0, size=size))   # nasty mixture
true_mu = 0.9 * 1.0 + 0.1 * 15.0

fig, axes = plt.subplots(1, 4, figsize=(13, 3))
for ax, n in zip(axes, [1, 5, 50, 500]):
    means = parent((20_000, n)).mean(axis=1) if n > 1 else parent((20_000,))
    ax.hist(means, bins=60, density=True)
    ax.axvline(true_mu, color="k", ls="--")
    ax.set_title(f"means of n={n}")
fig.suptitle("CLT: averages of an ugly distribution become a bell curve")
fig.tight_layout(); plt.show()

**The self-grading simulation.** One run gives both the estimate and its 95% CI. Then we *verify the verifier*: across 1,000 repetitions, the CI should contain the truth ~95% of the time.

In [ ]:
def estimate_with_ci(values):
    est = values.mean()
    se  = values.std(ddof=1) / np.sqrt(len(values))
    return est, se, (est - 1.96 * se, est + 1.96 * se)

r = np.random.default_rng(5)
vals = parent((50_000,))
est, se, ci = estimate_with_ci(vals)
print(f"estimate = {est:.4f} ± {1.96*se:.4f}   (true mean = {true_mu:.4f})")

covered = 0
for _ in range(1000):
    v = parent((5_000,))
    _, _, (lo, hi) = estimate_with_ci(v)
    covered += (lo <= true_mu <= hi)
print(f"CI coverage over 1000 reps: {covered/10:.1f}%  (should be ≈ 95%)")

**σ vs σ/√N — the classic confusion, in numbers.** Simulated NPVs: the spread of *outcomes* (risk of the world) vs the noise of the *estimated mean* (your computational precision). Factor of √N apart.

In [ ]:
r = np.random.default_rng(6)
npv = r.normal(52, 30, size=10_000)          # toy NPVs, $M
sd  = npv.std(ddof=1)
se  = sd / np.sqrt(len(npv))
print(f"σ (spread of possible outcomes / project risk):  ${sd:5.2f}M")
print(f"σ/√N (noise of your estimated mean):             ${se:5.2f}M")
print(f"ratio = {sd/se:.0f}  = √N = {np.sqrt(len(npv)):.0f}")

## Section 5 — Seeds, Reproducibility, and the Parallel Trap
> Explained in the guide: [Part 1 - where random numbers come from](https://statexampro.com/lessons/monte-carlo/first-principles/#seeds)


In [ ]:
a = np.random.default_rng(42).normal(size=5)
b = np.random.default_rng(42).normal(size=5)
c = np.random.default_rng(43).normal(size=5)
print("same seed  :", np.allclose(a, b))     # identical streams
print("other seed :", np.allclose(a, c))     # different stream

# THE TRAP: 4 'parallel workers' all seeded with 42 -> photocopied scenarios
workers_bad = [np.random.default_rng(42).uniform(size=3) for _ in range(4)]
print("\nBad parallel seeding (all workers identical):")
for w in workers_bad: print(" ", w.round(3))

# THE FIX: SeedSequence.spawn -> reproducible AND independent streams
ss = np.random.SeedSequence(42)
workers_good = [np.random.default_rng(s).uniform(size=3) for s in ss.spawn(4)]
print("Correct parallel seeding (independent streams):")
for w in workers_good: print(" ", w.round(3))

## Section 6 — From Uniform to Anything

**Inverse transform, by hand.** Exponential(λ): F(x) = 1 − e^(−λx), so F⁻¹(u) = −ln(1−u)/λ. We manufacture exponential samples from raw uniforms and check them against the true density.
> Explained in the guide: [Part 2 - from uniform to anything](https://statexampro.com/lessons/monte-carlo/recipe-and-projects/#sampling)


In [ ]:
r = np.random.default_rng(7)
lam = 0.5
u = r.uniform(size=200_000)
x = -np.log(1 - u) / lam                      # our hand-made exponential samples

grid = np.linspace(0, 15, 400)
plt.hist(x, bins=100, density=True, alpha=0.6, label="inverse-transform samples")
plt.plot(grid, lam * np.exp(-lam * grid), "k", lw=2, label="true Exp(0.5) density")
plt.xlim(0, 15); plt.legend(); plt.title("Inverse transform: uniforms → exponentials")
plt.show()
print(f"sample mean = {x.mean():.4f}   theory 1/λ = {1/lam:.4f}")

**Box–Muller.** Two uniforms in, two exact standard normals out — polar coordinates do the work. We verify with a QQ-plot against the true normal.

In [ ]:
r = np.random.default_rng(8)
u1, u2 = r.uniform(size=100_000), r.uniform(size=100_000)
z1 = np.sqrt(-2 * np.log(u1)) * np.cos(2 * np.pi * u2)
z2 = np.sqrt(-2 * np.log(u1)) * np.sin(2 * np.pi * u2)
z = np.concatenate([z1, z2])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(z, bins=100, density=True, alpha=0.6)
g = np.linspace(-4, 4, 300)
axes[0].plot(g, stats.norm.pdf(g), "k", lw=2); axes[0].set_title("Box–Muller samples vs N(0,1)")
stats.probplot(z[:5000], dist="norm", plot=axes[1]); axes[1].set_title("QQ-plot")
plt.tight_layout(); plt.show()
print(f"mean = {z.mean():+.4f}   sd = {z.std():.4f}   (want 0 and 1)")

**Sampling from data (empirical inverse CDF).** No formula needed: sorted data *is* a discretized inverse CDF. `np.quantile` at uniform percentiles ≈ draws that look like your dataset.

In [ ]:
r = np.random.default_rng(9)
history = r.lognormal(0.03, 0.25, size=800)        # pretend: 800 observed annual growth factors
new_draws = np.quantile(history, r.uniform(size=100_000), method="linear")

plt.hist(history, bins=40, density=True, alpha=0.5, label="observed data (n=800)")
plt.hist(new_draws, bins=80, density=True, histtype="step", lw=2, label="simulated from empirical CDF")
plt.legend(); plt.title("Simulating 'draws that look like my data'"); plt.show()

## Section 8 — Project 1 (Valuation): DCF as a Distribution

Five-year DCF of a subscription business. Uncertain inputs: revenue growth = a **persistent** component ~ Normal(12%, 4%) drawn once per world (good businesses keep being good) plus annual noise ~ Normal(0, 3%); EBITDA margin ~ Triangular(22%, 30%, 35%); exit multiple ~ Triangular(6, 8, 11). WACC fixed at 11% (a *price*, not a state of the world — see guide §8.5). Rows = worlds, columns = years.
> Explained in the guide: [Part 2 - project 1: the DCF as a distribution](https://statexampro.com/lessons/monte-carlo/recipe-and-projects/#dcf)


In [ ]:
def simulate_dcf(N, rng, years=5, rev0=10.0, wacc=0.11):
    g_bar = rng.normal(0.12, 0.04, size=(N, 1))              # persistent growth: drawn once per world
    g     = g_bar + rng.normal(0, 0.03, size=(N, years))     # plus independent annual noise
    rev  = rev0 * np.cumprod(1 + g, axis=1)                  # compounded revenue
    m    = rng.triangular(0.22, 0.30, 0.35, size=(N, 1))     # margin per world
    ebitda = rev * m
    disc = (1 + wacc) ** -np.arange(1, years + 1)            # discount factors
    mult = rng.triangular(6, 8, 11, size=N)                  # exit multiple
    return ebitda @ disc + mult * ebitda[:, -1] * disc[-1]   # PV(flows) + PV(exit)

rng8 = np.random.default_rng(2026)
value = simulate_dcf(100_000, rng8)

est, se, ci = value.mean(), value.std(ddof=1)/np.sqrt(len(value)), None
print(f"Expected value : ${est:.2f}M ± ${1.96*se:.2f}M (95% CI on the mean)")
print(f"Spread of value: σ = ${value.std(ddof=1):.2f}M   <- risk of the deal, NOT simulation noise")

In [ ]:
p = np.percentile(value, [5, 25, 50, 75, 95])
plt.hist(value, bins=120, density=True, alpha=0.7)
for q, lbl in zip(p, ["P5","P25","P50","P75","P95"]):
    plt.axvline(q, color="k", ls=":", lw=1)
    plt.text(q, plt.ylim()[1]*0.92, lbl, ha="center", fontsize=8)
plt.axvline(value.mean(), color="r", ls="--", label=f"mean ${value.mean():.1f}M")
plt.xlabel("enterprise value, $M"); plt.legend()
plt.title("Distribution of value (right skew: multiplicative arithmetic at work)")
plt.show()

print("percentiles P5/P25/P50/P75/P95 ($M):", p.round(1))
print(f"P(value < $30M) = {(value < 30).mean():.1%}")
print(f"P(value > $40M asking price) = {(value > 40).mean():.1%}")

**Jensen's gap.** Plug the *average* inputs into the model once (the deterministic 'base case') and compare with the true expected value. Compounding a persistent uncertain growth rate is **convex**: E[(1+g)^5] > (1+E[g])^5, so the base case *understates* the expected value. (Fun check: make the growth draws fully independent across years and the gap vanishes — expectation of a product of independent terms is the product of expectations. Nonlinearity only bites when uncertainty persists.)

In [ ]:
years, rev0, wacc = 5, 10.0, 0.11
g_det = 0.12; m_det = (0.22 + 0.30 + 0.35) / 3; mult_det = (6 + 8 + 11) / 3
rev_det = rev0 * (1 + g_det) ** np.arange(1, years + 1)
ebitda_det = rev_det * m_det
disc = (1 + wacc) ** -np.arange(1, years + 1)
v_det = ebitda_det @ disc + mult_det * ebitda_det[-1] * disc[-1]
print(f"deterministic 'base case' (mean inputs): ${v_det:.2f}M")
print(f"true expected value (simulation):        ${value.mean():.2f}M")
print(f"Jensen gap: base case understates E[value] by ${value.mean() - v_det:.2f}M "
      f"({(value.mean()/v_det - 1):.1%}) — compounding convexity at work")

**Crude tornado: which input owns the uncertainty?** Freeze each input at its central value, one at a time; whichever freeze shrinks the output spread most is the input that matters most — and where analyst hours should go.

In [ ]:
def simulate_dcf_frozen(N, rng, freeze, years=5, rev0=10.0, wacc=0.11):
    if freeze == "growth":
        g = np.full((N, years), 0.12)
    else:
        g = rng.normal(0.12, 0.04, (N, 1)) + rng.normal(0, 0.03, (N, years))
    m    = np.full((N, 1), 0.29)     if freeze=="margin"   else rng.triangular(0.22,0.30,0.35,(N,1))
    mult = np.full(N, 25/3)          if freeze=="multiple" else rng.triangular(6, 8, 11, N)
    rev  = rev0 * np.cumprod(1 + g, axis=1)
    ebitda = rev * m
    disc = (1 + wacc) ** -np.arange(1, years + 1)
    return ebitda @ disc + mult * ebitda[:, -1] * disc[-1]

base_sd = value.std()
print(f"full model output sd: ${base_sd:.2f}M\n")
for f in ["growth", "margin", "multiple"]:
    sd_f = simulate_dcf_frozen(100_000, np.random.default_rng(2026), f).std()
    print(f"freeze {f:<8s} -> sd ${sd_f:5.2f}M   ({f} owned ~{(1 - (sd_f/base_sd)**2):.0%} of variance)")

## Section 9 — Project 2 (Biostat/DS): Power by Simulation

One function = one complete simulated study, analysis included, returning one bit: *did we detect the effect?* Power = mean of bits.
> Explained in the guide: [Part 2 - project 2: power by simulation](https://statexampro.com/lessons/monte-carlo/recipe-and-projects/#power)


In [ ]:
def one_study(n_per_group, effect, sd, rng, dropout=0.0, skewed=False):
    if skewed:   # right-skewed outcome with the same mean shift and sd
        base = rng.lognormal(0, 0.8, size=(2, n_per_group))
        base = (base - base.mean()) / base.std() * sd
        a, b = base[0], base[1] + effect
    else:
        a = rng.normal(0,      sd, n_per_group)
        b = rng.normal(effect, sd, n_per_group)
    if dropout > 0:                              # each subject leaves w.p. dropout
        a = a[rng.uniform(size=len(a)) > dropout]
        b = b[rng.uniform(size=len(b)) > dropout]
    if min(len(a), len(b)) < 3:
        return False
    return stats.ttest_ind(a, b).pvalue < 0.05

def power(n, n_sim, rng, **kw):
    hits = np.mean([one_study(n, rng=rng, **kw) for _ in range(n_sim)])
    se = np.sqrt(hits * (1 - hits) / n_sim)
    return hits, se

# VALIDATION against the closed-form answer (always certify the machinery first)
rng9 = np.random.default_rng(99)
p_sim, se = power(64, 5000, rng9, effect=0.5, sd=1.0)
from scipy.stats import norm
z = 0.5 / np.sqrt(2 / 64)                        # textbook two-sample power
p_theory = norm.cdf(z - 1.96) + norm.cdf(-z - 1.96)
print(f"simulated power : {p_sim:.3f} ± {1.96*se:.3f}")
print(f"analytical power: {p_theory:.3f}   -> machinery certified")

**The power curve** — the actual deliverable of a sample-size analysis — first for the textbook design, then for the *realistic* one (15% dropout, skewed outcome), where no formula exists.

In [ ]:
ns = np.arange(10, 151, 10)
rng9 = np.random.default_rng(100)
curve_clean, curve_real, errbars = [], [], []
for n in ns:
    p1, s1 = power(n, 2000, rng9, effect=0.5, sd=1.0)
    p2, s2 = power(n, 2000, rng9, effect=0.5, sd=1.0, dropout=0.15, skewed=True)
    curve_clean.append(p1); curve_real.append(p2); errbars.append(1.96 * s2)

plt.errorbar(ns, curve_clean, fmt="o-", label="textbook design")
plt.errorbar(ns, curve_real, yerr=errbars, fmt="s-", label="15% dropout + skewed outcome")
plt.axhline(0.80, color="k", ls="--", label="80% target")
plt.xlabel("n per group"); plt.ylabel("power"); plt.legend()
plt.title("Power curves by simulation (error bars = MC noise, Ch.4)")
plt.show()

n80_clean = ns[np.searchsorted(curve_clean, 0.80)]
n80_real  = ns[np.searchsorted(curve_real, 0.80)]
print(f"n for 80% power — textbook: ~{n80_clean}/group,  realistic: ~{n80_real}/group "
      f"({n80_real/n80_clean - 1:+.0%} enrollment)")

**The lie detector: Type I error.** Same machinery, `effect=0` — the rejection rate should be α = 0.05. Then we audit a genuinely bad practice: *peeking* at the p-value every 20 subjects and stopping at the first significant result.

In [ ]:
rng9 = np.random.default_rng(101)
t1, se = power(15, 5000, rng9, effect=0.0, sd=1.0, skewed=True)
print(f"Type I error, t-test on skewed data, n=15/group: {t1:.3f} ± {1.96*se:.3f}  (nominal 0.05)")

def peeking_study(rng, max_n=200, step=20):
    a = rng.normal(0, 1, max_n); b = rng.normal(0, 1, max_n)   # NO real effect
    for n in range(step, max_n + 1, step):
        if stats.ttest_ind(a[:n], b[:n]).pvalue < 0.05:
            return True                                        # 'significant!' -> stop & publish
    return False

fp = np.mean([peeking_study(rng9) for _ in range(3000)])
print(f"False-positive rate WITH peeking every 20 subjects: {fp:.1%}  (nominal 5%!)")

## Exercises

1. **(π, but a circle of your own.)** Estimate the area under `sin(x)` on [0, π] by darts in the rectangle [0, π] × [0, 1]. Attach a 95% CI. Check against the exact answer (2).
2. **(√N tax.)** In Section 8, how large must N be for the 95% CI on the mean value to be ±\$0.02M? Predict from the σ/√N formula first, then verify by running.
3. **(Break Sin 4.)** In `simulate_dcf`, make the exit multiple *depend* on realized growth: `mult = 8 + 15*(rev[:, -1]/rev0**1 - (1.12)**5) + noise` (or any link you prefer). How do P5 and P95 move relative to the independent version, and why?
4. **(Inverse transform.)** Derive F⁻¹ for the Pareto distribution F(x) = 1 − (x_m/x)^α, generate samples, and watch the running mean for α = 1.5 vs α = 0.8. Which chapter-3 pathology appears, and why?
5. **(Method audit.)** Estimate the *coverage* of the standard 95% CI for a mean when data are Exp(1) and n = 10. Is it really 95%? At what n does it become acceptable?

*Sketch solutions are in the final cell — but the guide's Chapter 0, rule 3, applies.*

In [ ]:
# --- Exercise solution sketches (run after honest attempts) -------------
r = np.random.default_rng(777)

# 1) integral of sin on [0, pi]
N = 400_000
xs, ys = r.uniform(0, np.pi, N), r.uniform(0, 1, N)
hit = ys <= np.sin(xs)
area = np.pi * hit.mean()
se = np.pi * hit.std(ddof=1) / np.sqrt(N)
print(f"1) area = {area:.4f} ± {1.96*se:.4f}   (exact 2)")

# 2) N for ±$0.02M:  1.96*sigma/sqrt(N) = 0.02  ->  N = (1.96*sigma/0.02)^2
sigma = value.std(ddof=1)
print(f"2) need N ≈ {(1.96*sigma/0.02)**2:,.0f}")

# 5) CI coverage for Exp(1), n=10
cover = 0
for _ in range(20_000):
    v = r.exponential(1.0, 10)
    half = 1.96 * v.std(ddof=1) / np.sqrt(10)
    cover += abs(v.mean() - 1.0) <= half
print(f"5) coverage at n=10: {cover/200:.1f}% (skew makes the CLT approximation optimistic)")

---
**Next:** Part II, the Pro guide — geometric Brownian motion and option pricing vs Black–Scholes, correlated inputs (Cholesky, copulas), the variance-reduction arsenal, importance sampling for rare events, and quasi–Monte Carlo.

---

### Back to the guide

[The four parts](https://statexampro.com/lessons/monte-carlo/) &middot; [Part 1](https://statexampro.com/lessons/monte-carlo/first-principles/) &middot; [Part 2](https://statexampro.com/lessons/monte-carlo/recipe-and-projects/) &middot; [Part 3](https://statexampro.com/lessons/monte-carlo/paths-and-options/) &middot; [Part 4](https://statexampro.com/lessons/monte-carlo/variance-reduction/)

The guide and these notebooks are free, and stay that way because of the app: **Stat Exam Pro** drills the
exam-style questions behind these ideas - the traps, with worked answers, in English and French.
[Get it on the App Store](https://apps.apple.com/us/app/stat-exam-pro/id6755912771).
